# Automization of Data Download

In [12]:
import time
import requests
import pandas as pd
import altair as alt

## Getting Markets

In [13]:
# Reading CSV with market info
market_receipt = pd.read_csv("other/market_list.csv")
market_receipt

,Name,Resolution,Category,Size (Vol),Resolution Date,Slug,Out Prefix
0,Trump’s travel ban removed by January 31?,“No”,Pop-Culture,$10k Vol.,"Jan 31, 2026",trumps-travel-ban-removed-by-january-31,trumps-31
1,Young Thug found guilty of racketeering,"""Yes""",Politics,$15k Vol.,"Jul 1, 2024",young-thug-found-guilty-of-racketeering,young-racketeering
2,Fact Check: is LA U-Haul attack perp a U.S. Ci...,"""No""",Geopolitics,$26k Vol.,"Mar 1, 2026",fact-check-is-la-u-haul-attack-perp-a-us-citizen,fact-citizen
3,Apple AirTags 2 priced below $30 USD at release,"""Yes""",Technology,$61k Vol.,"Jan 26, 2026",apple-airtags-2-priced-below-30-usd-at-release,apple-release
4,Nothing Ever Happens: February,"""Something""",Geopolitics,$86k Vol.,"Feb 28, 2026",nothing-ever-happens-february-152,nothing-152
5,X relaunches Vine in 2025,"""No""",Technology,$150k Vol.,"Jan 4, 2026",x-relaunches-vine-in-2025,x-2025
6,Will GTA VI be released in 2025,"""No""",Pop-Culture,$1.4m Vol.,"Jan 1, 2026",gta-vi-released-in-2025,gta-2025
7,Canada vs USA,"""USA""",Sports,$2.98m Vol.,"Feb 22, 2026",mwoh-can-usa-2026-02-22,mwoh-22
8,Thunder vs Nuggets,"""Thunder""",Sports,$7.01m Vol.,"Feb 26, 2026",nba-den-okc-2026-02-27,nba-27
9,US Government Shutdown Saturday?,"""Yes""",Politics,$157m Vol.,"Feb 3, 2026",will-there-be-another-us-government-shutdown-b...,will-31


In [27]:
# Read Function
GAMMA = "https://gamma-api.polymarket.com"
DATA  = "https://data-api.polymarket.com"

def get_market(market_id: str = None, slug: str = None) -> dict:
    if (market_id is None) == (slug is None):
        raise ValueError("Provide either a market_id or slug.")

    if market_id:
        r = requests.get(f"{GAMMA}/markets/{market_id}", timeout=30)
    else:
        r = requests.get(f"{GAMMA}/markets/slug/{slug}", timeout=30)

    r.raise_for_status()
    return r.json()


# def fetch_all_trades(condition_id: str, limit: int = 500):
#     all_trades = []
#     before = None
#     while True:
#         params = {
#             "market": condition_id,
#             "limit": limit,
#         }
#         if before:
#             params["before"] = before
#         r = requests.get(f"{DATA}/trades", params=params, timeout=30)
#         r.raise_for_status()
#         trades = r.json()
#         if not trades:
#             break
#         all_trades.extend(trades)
#         print(
#             f"Fetched {len(all_trades)} trades so far... | "
#             f"batch newest: {trades[0]['timestamp']} | "
#             f"batch oldest: {trades[-1]['timestamp']} | "
#             f"next before: {before}"
#         )
#         before = trades[-1]["timestamp"] - 1
#         if len(trades) < limit:
#             break
#         time.sleep(0.05)
#     return all_trades

def fetch_all_trades(condition_id: str, limit: int = 500, max_offset: int = 3000):
    all_trades = []
    offset = 0

    while offset <= max_offset:
        params = {
            "market": condition_id,
            "limit": limit,
            "offset": offset,
            "takerOnly": True,
        }

        r = requests.get(f"{DATA}/trades", params=params, timeout=30)

        # clean stop at cap
        if r.status_code == 400:
            print(f"Hit cap at offset={offset}. Returning {len(all_trades)} trades.")
            break

        r.raise_for_status()
        batch = r.json()

        if not batch:
            break

        all_trades.extend(batch)
        print(f"Fetched {len(all_trades)} trades so far... (offset={offset})")

        if len(batch) < limit:
            break

        offset += limit
        time.sleep(0.02)

    return all_trades


def download_market_trades(market_id: str = None,
                           slug: str = None,
                           out_prefix: str = "polymarket"):

    market = get_market(market_id=market_id, slug=slug)
    condition_id = market["conditionId"]

    trades = fetch_all_trades(condition_id)
    trades_df = pd.DataFrame(trades)

    filename = f"market_csvs/{out_prefix}_trades.csv"
    trades_df.to_csv(filename, index=False)

    print(f"Saved {len(trades_df)} trades to {filename}")

    return trades_df

In [28]:
# Downloading all markets

for _, row in market_receipt.iterrows():
    s = row["Slug"]
    op = row["Out Prefix"]
    download_market_trades(slug=s, out_prefix=op)

Fetched 150 trades so far... (offset=0)
Saved 150 trades to market_csvs/trumps-31_trades.csv
Fetched 155 trades so far... (offset=0)
Saved 155 trades to market_csvs/young-racketeering_trades.csv
Fetched 467 trades so far... (offset=0)
Saved 467 trades to market_csvs/fact-citizen_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 857 trades so far... (offset=500)
Saved 857 trades to market_csvs/apple-release_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 952 trades so far... (offset=500)
Saved 952 trades to market_csvs/nothing-152_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 1000 trades so far... (offset=500)
Fetched 1254 trades so far... (offset=1000)
Saved 1254 trades to market_csvs/x-2025_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 1000 trades so far... (offset=500)
Fetched 1500 trades so far... (offset=1000)
Fetched 2000 trades so far... (offset=1500)
Fetched 2500 trades so far... (offset=2000)
Fetched 3000 trades so far... (offset=25